# Training a Computer Vision Model on the Historical Periodical "Die Bombe"

For training a  I have annotated 508 pages of "Die Bombe" with the Annotation Tool [Label Studio](https://labelstud.io/) and created a validation set consisting of 106 pages not contained in the training data.

##Time-period of the periodical
The periodical was publicated in vienna over a 50 year period (1871-1925), with issues appearing once per week in the starting period until a fade-out in mid-1917, where it started to be published three times a month, then only two times per month in 1920 and once per month at the end of 1923 until it's last publication on the 01.02.1925.

##Training and Validation Data
As I have had the chance to get an overview of the periodical before the start of my masters thesis, I already got a first visual impression, which time-period contained the recurring characters I was interested in. Most of them seemed to be contained in the issues before 1900. For this reason, I gathered issues from the periods 1871 till 1900, which make up 2/3 of the training data, while 1/3 consists of issues taken from the period of 1900 till 1925. The same treatment was used for approaching the gathering of validation data, while also making sure there is no overlap with the training-data.

##Annotation of the Datasets
The datasets were annotated according to the lables used by the [Newspaper Navigator](https://labs.loc.gov/work/experiments/newspaper-navigator/) Model of the Chronicling America Project conducted by the Library of Congress.

The labels are the following:
 - 0: Advertisement
 - 1: Comic
 - 2: Editorial Cartoon
 - 3: Headline
 - 4: Illustration
 - 5: Map
 - 6: Photograph

 For annotation BoundingBoxes were used instead of polygone regions and for training the annotated data was exported from Label Studio in COCO-format.

 For further explanations on the annotation schema please review the paper of the Newspaper Navigator and my [GitHub-Repository](https://github.com/lisagollner/Building-Character_Code.git) for further adjustements done by myself.

##Environment
This notebook was created with the help of ChatGPT-5.5 and is supposed to be used in a Google Colab/Google Drive environment.

## 1) Installing Dependencies and Importing Libraries

In [ ]:
# ==================================
# Install (Colab) + Restart Runtime
# ==================================

!pip -q install -U matplotlib opencv-python pycocotools
!pip -q uninstall -y torch torchvision torchaudio
!pip -q install torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121
!pip -q install ultralytics==8.2.0

import torch, ultralytics
print("torch:", torch.__version__)
print("ultralytics:", ultralytics.__version__)

# Reminder for restarting the runtime
print("✅ Installed/updated packages. Please restart the runtime ONCE: Runtime → Restart runtime, then run from cell 2 onwards.")


In [ ]:
# Checking whether GPU is accesible
!nvidia-smi || true

import torch, platform
print("torch:", torch.__version__, "cuda available?", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
print(platform.platform())


In [ ]:
# =================================================
# Imports (to be run after restarting the runtime)
# =================================================
import os
import json
import random
import hashlib
import platform
import collections
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

from PIL import Image as PILImage
from IPython.display import Image as DisplayImage, display

import ultralytics
from ultralytics import YOLO

## 2) Mount Google Drive + Setting Configurations
### Data contained in the "dataset" directory containing training and validations sets should be structured as follows:

```
  images/
    train/   # 508 training page images
    val/     # 106 validation page images
  annotations/
    instances_train.json
    instances_val.json
  labels/
    train/   # 508 txt-files
    val/     # 106 txt-files
```


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Model"
WEIGHTS_DIR = f"{PROJECT_DIR}/weights"
DATASET_DIR = f"{PROJECT_DIR}/training_dataset"

# Create folders if missing
os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATASET_DIR:", DATASET_DIR)
print("WEIGHTS_DIR:", WEIGHTS_DIR)


## 4) Create or load `data.yaml` for Ultralytics

Ultralytics uses the YOLO-format `.txt` annotation files in
`labels/train` and `labels/val`. The COCO JSON files are retained
for dataset documentation or additional checks but are not used
directly by this training call.

In [ ]:
DATA_YAML_PATH = Path(PROJECT_DIR) / "data.yaml"

DATA_YAML = f"""
path: {DATASET_DIR}
train: images/train
val: images/val
names:
  0: Advertisement
  1: Comic
  2: Editorial Cartoon
  3: Headline
  4: Illustration
  5: Map
  6: Photograph
"""

if DATA_YAML_PATH.exists():
    print(f"✅ data.yaml already exists at: {DATA_YAML_PATH}")
else:
    DATA_YAML_PATH.write_text(DATA_YAML.strip() + "\n")
    print(f"🆕 Created new data.yaml at: {DATA_YAML_PATH}")

print("\n--- data.yaml content ---")
print(DATA_YAML_PATH.read_text())


## 5) YOLO Label Sanity Check

In [ ]:
# ============================================================
# Verify class IDs in YOLO-format training and validation labels
# ============================================================

EXPECTED_CLASS_IDS = set(range(7))

for split in ["train", "val"]:
    labels_dir = Path(DATASET_DIR) / "labels" / split

    if not labels_dir.exists():
        raise FileNotFoundError(
            f"Missing label directory: {labels_dir}"
        )

    used_class_ids = set()
    invalid_lines = []

    label_files = sorted(labels_dir.glob("*.txt"))

    if not label_files:
        raise FileNotFoundError(
            f"No YOLO label files found in: {labels_dir}"
        )

    for label_path in label_files:
        for line_number, line in enumerate(
            label_path.read_text(
                encoding="utf-8"
            ).splitlines(),
            start=1,
        ):
            parts = line.strip().split()

            if not parts:
                continue

            if len(parts) != 5:
                invalid_lines.append(
                    (
                        label_path.name,
                        line_number,
                        line,
                    )
                )
                continue

            used_class_ids.add(int(parts[0]))

    unexpected_ids = used_class_ids - EXPECTED_CLASS_IDS
    missing_ids = EXPECTED_CLASS_IDS - used_class_ids

    print(f"\n{split.upper()}")
    print(f"Label files:     {len(label_files)}")
    print(f"Class IDs used:  {sorted(used_class_ids)}")

    if unexpected_ids:
        raise ValueError(
            f"Unexpected class IDs in {split}: "
            f"{sorted(unexpected_ids)}"
        )

    if invalid_lines:
        raise ValueError(
            f"Found {len(invalid_lines)} malformed lines "
            f"in the {split} labels. First examples: "
            f"{invalid_lines[:5]}"
        )

    if missing_ids:
        print(
            "Warning: these configured classes do not occur "
            f"in {split}: {sorted(missing_ids)}"
        )


## 6) Loading a Pretrained YOLOv8 Model
Model used for training: 'yolov8m.pt'

In [ ]:
# =======================================
# Run Config + Persistent Logging (Drive)
# =======================================

RUNS_PROJECT_DIR = "/content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Model/runs"
DATA_YAML_PATH = f"{PROJECT_DIR}/data.yaml"

# Keep RUN_NAME stable if you want to resume the same run after interruptions.
RUN_NAME = "yolov8m_final-pipeline_run-001"
RUN_DIR = Path(RUNS_PROJECT_DIR) / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

CFG = {
    "model_base": "yolov8m.pt",
    "data": DATA_YAML_PATH,
    "epochs": 100,
    "imgsz": 1280,
    "batch": 4,
    "workers": 2,
    "patience": 20,

    # training settings
    "optimizer": "SGD",
    "lr0": 0.01,
    "cos_lr": True,
    "mosaic": 1.0,
    "cache": False,

    # reproducibility
    "seed": 0,
    "deterministic": True,
}

def sha256_file(p: str) -> str:
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

env = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "ultralytics": ultralytics.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}

try:
    env["data_yaml_sha256"] = sha256_file(DATA_YAML_PATH)
except Exception as e:
    env["data_yaml_sha256_error"] = str(e)

# per-run config
run_config_path = RUN_DIR / "run_config.json"
run_config_path.write_text(
    json.dumps({"cfg": CFG, "env": env, "run_name": RUN_NAME, "project_dir": RUNS_PROJECT_DIR}, indent=2),
    encoding="utf-8"
)
print("Wrote:", run_config_path)

# central registry
registry_path = Path(RUNS_PROJECT_DIR) / "run_registry.csv"
header = ("timestamp,run_name,model_base,data,epochs,imgsz,batch,workers,patience,optimizer,lr0,cos_lr,mosaic,cache,seed,deterministic,"
          "torch,ultralytics,cuda_device,data_yaml_sha256\n")
row = (f"{env['timestamp']},{RUN_NAME},{CFG['model_base']},{CFG['data']},{CFG['epochs']},{CFG['imgsz']},{CFG['batch']},{CFG['workers']},"
       f"{CFG['patience']},{CFG['optimizer']},{CFG['lr0']},{CFG['cos_lr']},{CFG['mosaic']},{CFG['cache']},{CFG['seed']},{CFG['deterministic']},"
       f"{env['torch']},{env['ultralytics']},{env['cuda_device']},{env.get('data_yaml_sha256','')}\n")

if not registry_path.exists():
    registry_path.write_text(header, encoding="utf-8")
with open(registry_path, "a", encoding="utf-8") as f:
    f.write(row)
print("Updated registry:", registry_path)

#for disabling Weigths & Biases logging
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"

from ultralytics import settings

settings.update({"wandb": False})
print(settings["wandb"])

# =====================================
# Train (auto-resume if last.pt exists)
# =====================================
RUNS_PROJECT_DIR = "/content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Model/runs"
RUN_NAME = "yolov8m_final-pipeline_run-001"

RUN_DIR = Path(RUNS_PROJECT_DIR) / RUN_NAME
WEIGHTS_DIR = RUN_DIR / "weights"
LAST_PT = WEIGHTS_DIR / "last.pt"
BEST_PT = WEIGHTS_DIR / "best.pt"

# Load the prepared configurations
cfg_path = RUN_DIR / "run_config.json"
assert cfg_path.exists(), f"Missing {cfg_path}. Run cell (7a) first."
CFG = json.loads(cfg_path.read_text(encoding="utf-8"))["cfg"]

device = 0 if torch.cuda.is_available() else "cpu"

if LAST_PT.exists():
    print(f"✅ Found checkpoint: {LAST_PT}")
    print("➡️ Resuming training from last.pt (safe after interruptions).")
    model = YOLO(str(LAST_PT))
    resume_flag = True
else:
    print("ℹ️ No last.pt found. Starting fresh from:", CFG["model_base"])
    model = YOLO(CFG["model_base"])
    resume_flag = False

results = model.train(
    data=CFG["data"],
    epochs=CFG["epochs"],
    imgsz=CFG["imgsz"],
    batch=CFG["batch"],
    patience=CFG["patience"],
    device=device,
    workers=CFG["workers"],
    project=RUNS_PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True,
    pretrained=True,
    optimizer=CFG["optimizer"],
    lr0=CFG["lr0"],
    cos_lr=CFG["cos_lr"],
    mosaic=CFG["mosaic"],
    cache=CFG["cache"],
    seed=CFG["seed"],
    deterministic=CFG["deterministic"],
    resume=resume_flag,
)

print("Done. Checkpoints:")
print("  last:", LAST_PT, "exists=", LAST_PT.exists())
print("  best:", BEST_PT, "exists=", BEST_PT.exists())
results

# ===========================================
# Write run summary (params + outputs paths)
# ===========================================
summary = {
    "run_name": RUN_NAME,
    "run_dir": str(RUN_DIR),
    "best_pt": str(BEST_PT),
    "last_pt": str(LAST_PT),
    "config_json": str(run_config_path),
    "ultralytics_args_yaml": str(RUN_DIR / "args.yaml"),
    "ultralytics_results_csv": str(RUN_DIR / "results.csv"),
}

out_path = RUN_DIR / "RUN_SUMMARY.json"
out_path.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

print("Wrote:", out_path)
print(json.dumps(summary, indent=2))


## 7) Evaluation on Validation Dataset

In [ ]:
# ============================================================
# Load trained model and evaluation configuration
# ============================================================

RUNS_PROJECT_DIR = Path("/content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Model/runs")

RUN_NAME = "yolov8m_final-pipeline_run-001"

RUN_DIR = RUNS_PROJECT_DIR / RUN_NAME
CONFIG_PATH = RUN_DIR / "run_config.json"
BEST_CHECKPOINT = RUN_DIR / "weights" / "best.pt"

assert RUN_DIR.exists(), f"Training run not found: {RUN_DIR}"
assert CONFIG_PATH.exists(), f"Run configuration not found: {CONFIG_PATH}"
assert BEST_CHECKPOINT.exists(), f"Best checkpoint not found: {BEST_CHECKPOINT}"

CFG = json.loads(
    CONFIG_PATH.read_text(encoding="utf-8")
)["cfg"]

DEVICE = 0 if torch.cuda.is_available() else "cpu"

model = YOLO(str(BEST_CHECKPOINT))

print(f"Run:        {RUN_NAME}")
print(f"Checkpoint: {BEST_CHECKPOINT}")
print(f"Dataset:    {CFG['data']}")
print(f"Image size: {CFG['imgsz']}")
print(f"Batch size: {CFG['batch']}")
print(f"Device:     {DEVICE}")

In [ ]:
# ============================================================
# Evaluate best.pt on the validation split
# ============================================================

VALIDATION_RUN_NAME = f"{RUN_NAME}_validation"

metrics = model.val(
    data=CFG["data"],
    split="val",
    imgsz=CFG["imgsz"],
    batch=CFG["batch"],
    device=DEVICE,
    project=str(RUNS_PROJECT_DIR),
    name=VALIDATION_RUN_NAME,
    exist_ok=True,
    plots=True,
    save_json=False,
)

VALIDATION_OUTPUT_DIR = RUNS_PROJECT_DIR / VALIDATION_RUN_NAME

print("\nValidation completed.")
print(f"Results saved to: {VALIDATION_OUTPUT_DIR}")

print("\nOverall validation metrics:")
print(f"Precision:  {metrics.box.mp:.4f}")
print(f"Recall:     {metrics.box.mr:.4f}")
print(f"mAP@0.50:   {metrics.box.map50:.4f}")
print(f"mAP@0.50:95:{metrics.box.map:.4f}")

##8) Detailed class-specific Precision, Recall, and F1 evaluation

In [ ]:
# ============================================================
# 8.1 Prepare predictions and ground truth
#     for class-specific confidence-threshold analysis
# ============================================================

VAL_IMAGES_DIR = Path(DATASET_DIR) / "images" / "val"
VAL_LABELS_DIR = Path(DATASET_DIR) / "labels" / "val"

CUSTOM_EVAL_DIR = RUNS_PROJECT_DIR / f"{RUN_NAME}_custom_evaluation"
CUSTOM_EVAL_DIR.mkdir(parents=True, exist_ok=True)

PREDICTIONS_CSV = CUSTOM_EVAL_DIR / "raw_predictions_val.csv"
GROUND_TRUTH_CSV = CUSTOM_EVAL_DIR / "ground_truth_val.csv"

MIN_PREDICTION_CONFIDENCE = 0.001
PREDICTION_NMS_IOU = 0.7

assert VAL_IMAGES_DIR.exists(), (
    f"Validation images not found: {VAL_IMAGES_DIR}"
)
assert VAL_LABELS_DIR.exists(), (
    f"Validation labels not found: {VAL_LABELS_DIR}"
)


# ------------------------------------------------------------
# A. Generate raw validation predictions
# ------------------------------------------------------------

prediction_rows = []

prediction_results = model.predict(
    source=str(VAL_IMAGES_DIR),
    imgsz=CFG["imgsz"],
    conf=MIN_PREDICTION_CONFIDENCE,
    iou=PREDICTION_NMS_IOU,
    device=DEVICE,
    save=False,
    verbose=False,
)

for result in prediction_results:
    image_name = Path(result.path).name

    if result.boxes is None:
        continue

    boxes_xyxy = result.boxes.xyxy.cpu().numpy()
    class_ids = result.boxes.cls.cpu().numpy().astype(int)
    confidences = result.boxes.conf.cpu().numpy()

    for box, class_id, confidence in zip(
        boxes_xyxy,
        class_ids,
        confidences,
    ):
        x1, y1, x2, y2 = box

        prediction_rows.append(
            {
                "image": image_name,
                "class_id": class_id,
                "class_name": model.names[class_id],
                "confidence": float(confidence),
                "x1": float(x1),
                "y1": float(y1),
                "x2": float(x2),
                "y2": float(y2),
            }
        )

prediction_columns = [
    "image",
    "class_id",
    "class_name",
    "confidence",
    "x1",
    "y1",
    "x2",
    "y2",
]

pred_df = pd.DataFrame(
    prediction_rows,
    columns=prediction_columns,
)

pred_df.to_csv(PREDICTIONS_CSV, index=False)


# ------------------------------------------------------------
# B. Convert YOLO validation labels to pixel coordinates
# ------------------------------------------------------------

ground_truth_rows = []

image_paths_by_stem = {
    image_path.stem: image_path
    for image_path in VAL_IMAGES_DIR.iterdir()
    if image_path.suffix.lower()
    in {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
}

for label_path in sorted(VAL_LABELS_DIR.glob("*.txt")):
    image_path = image_paths_by_stem.get(label_path.stem)

    if image_path is None:
        print(
            f"Warning: no validation image found for "
            f"{label_path.name}"
        )
        continue

    with PILImage.open(image_path) as image:
        image_width, image_height = image.size

    lines = label_path.read_text(
        encoding="utf-8"
    ).splitlines()

    for line_number, line in enumerate(lines, start=1):
        parts = line.strip().split()

        if not parts:
            continue

        if len(parts) != 5:
            print(
                f"Warning: invalid label in "
                f"{label_path.name}, line {line_number}: "
                f"{line}"
            )
            continue

        class_id = int(parts[0])
        x_center, y_center, box_width, box_height = map(
            float,
            parts[1:],
        )

        x1 = (x_center - box_width / 2) * image_width
        y1 = (y_center - box_height / 2) * image_height
        x2 = (x_center + box_width / 2) * image_width
        y2 = (y_center + box_height / 2) * image_height

        ground_truth_rows.append(
            {
                "image": image_path.name,
                "class_id": class_id,
                "class_name": model.names[class_id],
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
            }
        )

ground_truth_columns = [
    "image",
    "class_id",
    "class_name",
    "x1",
    "y1",
    "x2",
    "y2",
]

gt_df = pd.DataFrame(
    ground_truth_rows,
    columns=ground_truth_columns,
)

gt_df.to_csv(GROUND_TRUTH_CSV, index=False)


# ------------------------------------------------------------
# C. Validate and summarize the evaluation data
# ------------------------------------------------------------

required_prediction_columns = {
    "image",
    "class_id",
    "class_name",
    "confidence",
    "x1",
    "y1",
    "x2",
    "y2",
}

required_ground_truth_columns = {
    "image",
    "class_id",
    "class_name",
    "x1",
    "y1",
    "x2",
    "y2",
}

missing_prediction_columns = (
    required_prediction_columns - set(pred_df.columns)
)
missing_ground_truth_columns = (
    required_ground_truth_columns - set(gt_df.columns)
)

if missing_prediction_columns:
    raise ValueError(
        "Prediction data is missing columns: "
        f"{sorted(missing_prediction_columns)}"
    )

if missing_ground_truth_columns:
    raise ValueError(
        "Ground-truth data is missing columns: "
        f"{sorted(missing_ground_truth_columns)}"
    )

print("Custom evaluation data prepared.")
print(f"Predicted boxes:    {len(pred_df)}")
print(f"Ground-truth boxes: {len(gt_df)}")
print(f"Predictions saved:  {PREDICTIONS_CSV}")
print(f"Ground truth saved: {GROUND_TRUTH_CSV}")

display(pred_df.head())
display(gt_df.head())

In [ ]:
# ============================================================
# Class-specific Precision, Recall, and F1 curves
# ============================================================

IOU_THRESHOLD = 0.50
CONFIDENCE_THRESHOLDS = np.linspace(0.0, 1.0, 1001)

CURVE_DATA_CSV = (
    CUSTOM_EVAL_DIR / "f1_curves_per_class.csv"
)
BEST_THRESHOLDS_CSV = (
    CUSTOM_EVAL_DIR / "best_thresholds_per_class.csv"
)
MATCHED_PREDICTIONS_CSV = (
    CUSTOM_EVAL_DIR / "matched_predictions.csv"
)

F1_PLOT_PNG = (
    CUSTOM_EVAL_DIR / "per_class_f1_curves.png"
)
F1_PLOT_PDF = (
    CUSTOM_EVAL_DIR / "per_class_f1_curves.pdf"
)


# ------------------------------------------------------------
# A. Remove an optional background class
# ------------------------------------------------------------

pred_eval_df = pred_df.copy()
gt_eval_df = gt_df.copy()

pred_eval_df = pred_eval_df[
    pred_eval_df["class_name"]
    .astype(str)
    .str.lower()
    .ne("background")
].copy()

gt_eval_df = gt_eval_df[
    gt_eval_df["class_name"]
    .astype(str)
    .str.lower()
    .ne("background")
].copy()

classes = sorted(
    set(gt_eval_df["class_name"].unique())
    | set(pred_eval_df["class_name"].unique())
)

if not classes:
    raise ValueError(
        "No evaluation classes were found."
    )

print("Classes included in the custom evaluation:")
for class_name in classes:
    print(f"  - {class_name}")


# ------------------------------------------------------------
# B. IoU calculation
# ------------------------------------------------------------

def calculate_iou(box_a, box_b):
    """
    Calculate intersection over union for two boxes in
    [x1, y1, x2, y2] format.
    """
    intersection_x1 = max(box_a[0], box_b[0])
    intersection_y1 = max(box_a[1], box_b[1])
    intersection_x2 = min(box_a[2], box_b[2])
    intersection_y2 = min(box_a[3], box_b[3])

    intersection_width = max(
        0.0,
        intersection_x2 - intersection_x1,
    )
    intersection_height = max(
        0.0,
        intersection_y2 - intersection_y1,
    )

    intersection_area = (
        intersection_width * intersection_height
    )

    box_a_area = (
        max(0.0, box_a[2] - box_a[0])
        * max(0.0, box_a[3] - box_a[1])
    )

    box_b_area = (
        max(0.0, box_b[2] - box_b[0])
        * max(0.0, box_b[3] - box_b[1])
    )

    union_area = (
        box_a_area
        + box_b_area
        - intersection_area
    )

    if union_area <= 0:
        return 0.0

    return intersection_area / union_area


# ------------------------------------------------------------
# C. Match every prediction once
# ------------------------------------------------------------

def match_predictions_for_class(
    class_predictions,
    class_ground_truth,
    iou_threshold,
):
    """
    Match predictions to ground-truth boxes once, in descending
    confidence order.

    Each ground-truth box may be matched only once.
    """

    class_predictions = (
        class_predictions
        .sort_values(
            "confidence",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    ground_truth_by_image = {
        image_name: group.reset_index(drop=True)
        for image_name, group
        in class_ground_truth.groupby("image")
    }

    matched_ground_truth = set()
    match_rows = []

    for _, prediction in class_predictions.iterrows():
        image_name = prediction["image"]

        prediction_box = [
            prediction["x1"],
            prediction["y1"],
            prediction["x2"],
            prediction["y2"],
        ]

        best_iou = 0.0
        best_ground_truth_key = None

        image_ground_truth = ground_truth_by_image.get(
            image_name
        )

        if image_ground_truth is not None:
            for ground_truth_index, ground_truth in (
                image_ground_truth.iterrows()
            ):
                ground_truth_key = (
                    image_name,
                    ground_truth_index,
                )

                if ground_truth_key in matched_ground_truth:
                    continue

                ground_truth_box = [
                    ground_truth["x1"],
                    ground_truth["y1"],
                    ground_truth["x2"],
                    ground_truth["y2"],
                ]

                iou = calculate_iou(
                    prediction_box,
                    ground_truth_box,
                )

                if iou > best_iou:
                    best_iou = iou
                    best_ground_truth_key = (
                        ground_truth_key
                    )

        is_true_positive = (
            best_ground_truth_key is not None
            and best_iou >= iou_threshold
        )

        if is_true_positive:
            matched_ground_truth.add(
                best_ground_truth_key
            )

        match_rows.append(
            {
                "image": image_name,
                "class_name": prediction["class_name"],
                "confidence": prediction["confidence"],
                "best_iou": best_iou,
                "is_true_positive": int(
                    is_true_positive
                ),
                "is_false_positive": int(
                    not is_true_positive
                ),
            }
        )

    return pd.DataFrame(match_rows)


matched_prediction_frames = []

for class_name in classes:
    class_predictions = pred_eval_df[
        pred_eval_df["class_name"] == class_name
    ]

    class_ground_truth = gt_eval_df[
        gt_eval_df["class_name"] == class_name
    ]

    class_matches = match_predictions_for_class(
        class_predictions=class_predictions,
        class_ground_truth=class_ground_truth,
        iou_threshold=IOU_THRESHOLD,
    )

    if not class_matches.empty:
        matched_prediction_frames.append(
            class_matches
        )

if matched_prediction_frames:
    matched_predictions_df = pd.concat(
        matched_prediction_frames,
        ignore_index=True,
    )
else:
    matched_predictions_df = pd.DataFrame(
        columns=[
            "image",
            "class_name",
            "confidence",
            "best_iou",
            "is_true_positive",
            "is_false_positive",
        ]
    )

matched_predictions_df.to_csv(
    MATCHED_PREDICTIONS_CSV,
    index=False,
)


# ------------------------------------------------------------
# D. Calculate metrics for all confidence thresholds
# ------------------------------------------------------------

curve_rows = []

for class_name in classes:
    class_matches = matched_predictions_df[
        matched_predictions_df["class_name"]
        == class_name
    ].sort_values(
        "confidence",
        ascending=False,
    )

    number_of_ground_truth_boxes = len(
        gt_eval_df[
            gt_eval_df["class_name"] == class_name
        ]
    )

    confidences = class_matches[
        "confidence"
    ].to_numpy()

    true_positive_flags = class_matches[
        "is_true_positive"
    ].to_numpy(dtype=int)

    false_positive_flags = class_matches[
        "is_false_positive"
    ].to_numpy(dtype=int)

    cumulative_true_positives = np.cumsum(
        true_positive_flags
    )

    cumulative_false_positives = np.cumsum(
        false_positive_flags
    )

    for confidence_threshold in CONFIDENCE_THRESHOLDS:
        number_retained = np.count_nonzero(
            confidences >= confidence_threshold
        )

        if number_retained == 0:
            true_positives = 0
            false_positives = 0
        else:
            true_positives = int(
                cumulative_true_positives[
                    number_retained - 1
                ]
            )
            false_positives = int(
                cumulative_false_positives[
                    number_retained - 1
                ]
            )

        false_negatives = (
            number_of_ground_truth_boxes
            - true_positives
        )

        precision_denominator = (
            true_positives + false_positives
        )

        precision = (
            true_positives / precision_denominator
            if precision_denominator > 0
            else 0.0
        )

        recall = (
            true_positives
            / number_of_ground_truth_boxes
            if number_of_ground_truth_boxes > 0
            else 0.0
        )

        f1_denominator = precision + recall

        f1 = (
            2 * precision * recall
            / f1_denominator
            if f1_denominator > 0
            else 0.0
        )

        curve_rows.append(
            {
                "class_name": class_name,
                "confidence_threshold": (
                    confidence_threshold
                ),
                "tp": true_positives,
                "fp": false_positives,
                "fn": false_negatives,
                "precision": precision,
                "recall": recall,
                "f1": f1,
            }
        )

curve_df = pd.DataFrame(curve_rows)

curve_df.to_csv(
    CURVE_DATA_CSV,
    index=False,
)


# ------------------------------------------------------------
# E. Select the optimal threshold for each class
# ------------------------------------------------------------

best_rows = []

for class_name in classes:
    class_curve = curve_df[
        curve_df["class_name"] == class_name
    ].copy()

    maximum_f1 = class_curve["f1"].max()

    best_candidates = class_curve[
        class_curve["f1"] == maximum_f1
    ]

    # When multiple thresholds produce the same maximum F1,
    # prefer the highest threshold.
    best_row = best_candidates.sort_values(
        "confidence_threshold",
        ascending=False,
    ).iloc[0]

    best_rows.append(
        {
            "class_name": class_name,
            "best_confidence_threshold": (
                best_row["confidence_threshold"]
            ),
            "best_f1": best_row["f1"],
            "precision_at_best_f1": (
                best_row["precision"]
            ),
            "recall_at_best_f1": (
                best_row["recall"]
            ),
            "tp": int(best_row["tp"]),
            "fp": int(best_row["fp"]),
            "fn": int(best_row["fn"]),
            "ground_truth_boxes": int(
                (
                    gt_eval_df["class_name"]
                    == class_name
                ).sum()
            ),
            "predicted_boxes": int(
                (
                    pred_eval_df["class_name"]
                    == class_name
                ).sum()
            ),
        }
    )

best_df = pd.DataFrame(best_rows)

best_df.to_csv(
    BEST_THRESHOLDS_CSV,
    index=False,
)

display(
    best_df.style.format(
        {
            "best_confidence_threshold": "{:.3f}",
            "best_f1": "{:.3f}",
            "precision_at_best_f1": "{:.3f}",
            "recall_at_best_f1": "{:.3f}",
        }
    )
)


# ------------------------------------------------------------
# F. Plot the per-class F1 curves
# ------------------------------------------------------------

number_of_classes = len(classes)
number_of_columns = min(3, number_of_classes)
number_of_rows = int(
    np.ceil(
        number_of_classes / number_of_columns
    )
)

figure, axes = plt.subplots(
    nrows=number_of_rows,
    ncols=number_of_columns,
    figsize=(
        5 * number_of_columns,
        4.2 * number_of_rows,
    ),
    sharex=True,
    sharey=True,
    squeeze=False,
)

axes = axes.flatten()

for axis, class_name in zip(axes, classes):
    class_curve = curve_df[
        curve_df["class_name"] == class_name
    ]

    best_result = best_df[
        best_df["class_name"] == class_name
    ].iloc[0]

    axis.plot(
        class_curve["confidence_threshold"],
        class_curve["f1"],
        linewidth=2,
    )

    axis.axvline(
        best_result[
            "best_confidence_threshold"
        ],
        linestyle="--",
        linewidth=1,
    )

    axis.scatter(
        best_result[
            "best_confidence_threshold"
        ],
        best_result["best_f1"],
        s=40,
    )

    axis.set_title(
        f"{class_name}\n"
        f"F1={best_result['best_f1']:.3f}, "
        f"confidence="
        f"{best_result['best_confidence_threshold']:.3f}"
    )

    axis.set_xlabel("Confidence threshold")
    axis.set_ylabel("F1 score")
    axis.set_xlim(0.0, 1.0)
    axis.set_ylim(0.0, 1.0)
    axis.grid(alpha=0.3)

for unused_axis in axes[number_of_classes:]:
    unused_axis.axis("off")

figure.suptitle(
    "Class-specific F1-confidence curves "
    f"at IoU ≥ {IOU_THRESHOLD:.2f}",
    fontsize=16,
)

figure.tight_layout(
    rect=[0, 0, 1, 0.95]
)

figure.savefig(
    F1_PLOT_PNG,
    dpi=300,
    bbox_inches="tight",
)
figure.savefig(
    F1_PLOT_PDF,
    bbox_inches="tight",
)

plt.show()

print("\nCustom evaluation completed.")
print(f"Matched predictions: {MATCHED_PREDICTIONS_CSV}")
print(f"Complete curves:     {CURVE_DATA_CSV}")
print(f"Best thresholds:     {BEST_THRESHOLDS_CSV}")
print(f"PNG plot:            {F1_PLOT_PNG}")
print(f"PDF plot:            {F1_PLOT_PDF}")

##Resources

 - Yaseen (2024): What is YOLOv8: an in-depth Exploration of the Internal Features of the next-generation Object Detector. [DOI:
https://doi.org/10.48550/arXiv.2408.15857]
 - Redmon et al. (2015): You Only Look Once: Unified, Real-Time Object Detection. [DOI: https://doi.org/10.48550/arXiv.1506.02640]